# PicoRV32 + CIM SoC — 宿主机环境准备

本 notebook 在**宿主机** (有 RISC-V GCC + Python) 上运行，完成：

1. 训练 Small MLP (784→16→10) 并量化导出
2. Python golden model 验证 20 张测试图
3. **批量编译 20 个 firmware.hex** (每张图一个)
4. 打包文件，准备复制到 VCS 虚拟机 和 PYNQ 板

环境要求：
- RISC-V GCC (riscv64-elf-gcc 或 riscv64-unknown-elf-gcc)
- Python 3 + PyTorch + torchvision

VCS 虚拟机**不需要**任何编译工具，只需要 VCS。

**目录关系说明：**
```
项目根目录/
├── sw/                          ← 本 notebook 所在目录 (CWD)
│   ├── prepare_picorv32_env.ipynb
│   └── fw_hex_batch/            ← 本 notebook 输出目录
├── picorv32/fw/                 ← 固件源码 + 构建工具
│   ├── Makefile, firmware.c, start.S, firmware.lds
│   ├── gen_fw_data.py, small_mlp_quantize.py
│   └── small_mlp_data/          ← 模型训练数据 (由 small_mlp_quantize.py 生成)
└── bitstream&hwh/checkpoint2/   ← bitstream
```

## 0. 路径设置

In [1]:
import os, sys, subprocess, shutil, struct
import numpy as np

# ---- 本 notebook 从 sw/ 目录运行 ----
NOTEBOOK_DIR = os.path.abspath(".")
PROJ_ROOT    = os.path.dirname(NOTEBOOK_DIR)                # 项目根目录
FW_DIR       = os.path.join(PROJ_ROOT, "picorv32", "fw")    # 固件源码目录
DATA_DIR     = os.path.join(FW_DIR, "small_mlp_data")       # 模型数据 (绝对路径)
DATA_DIR_REL = "small_mlp_data"                              # make 用的相对路径 (相对于 FW_DIR)
OUT_DIR      = os.path.join(NOTEBOOK_DIR, "fw_hex_batch")   # 本 notebook 输出目录
N_IMAGES     = 200

print(f"Notebook CWD:  {NOTEBOOK_DIR}")
print(f"Project root:  {PROJ_ROOT}")
print(f"Firmware dir:  {FW_DIR}")
print(f"Data dir:      {DATA_DIR}")
print(f"Output dir:    {OUT_DIR}")
print()

# 验证关键文件存在
for f in ["firmware.c", "Makefile", "start.S", "firmware.lds", "gen_fw_data.py"]:
    p = os.path.join(FW_DIR, f)
    print(f"  {'✓' if os.path.exists(p) else '✗'} {f}")
    assert os.path.exists(p), f"缺少关键文件: {p}"

Notebook CWD:  /home/jiao/git/INT8-CIM-of-jiao/sw
Project root:  /home/jiao/git/INT8-CIM-of-jiao
Firmware dir:  /home/jiao/git/INT8-CIM-of-jiao/picorv32/fw
Data dir:      /home/jiao/git/INT8-CIM-of-jiao/picorv32/fw/small_mlp_data
Output dir:    /home/jiao/git/INT8-CIM-of-jiao/sw/fw_hex_batch

  ✓ firmware.c
  ✓ Makefile
  ✓ start.S
  ✓ firmware.lds
  ✓ gen_fw_data.py


## 1. 检查工具链

In [2]:
# RISC-V GCC
rv_gcc = None
cross_prefix = None
for prefix in ["riscv64-elf-", "riscv64-unknown-elf-", "riscv32-unknown-elf-"]:
    if shutil.which(f"{prefix}gcc"):
        rv_gcc = f"{prefix}gcc"
        cross_prefix = prefix
        break

if rv_gcc:
    ver = subprocess.check_output([rv_gcc, "--version"], text=True).split("\n")[0]
    print(f"✓ RISC-V GCC: {ver}")
    print(f"  CROSS prefix: {cross_prefix}")
else:
    print("✗ RISC-V GCC 未找到!")
    print("  Arch:   sudo pacman -S riscv64-elf-gcc")
    print("  Debian: sudo apt install gcc-riscv64-unknown-elf")

# Python
try:
    import torch; print(f"✓ PyTorch {torch.__version__}")
except ImportError:
    print("✗ PyTorch: pip install torch torchvision")

try:
    from torchvision import datasets; print("✓ torchvision")
except ImportError:
    print("✗ torchvision: pip install torchvision")

✓ RISC-V GCC: riscv64-elf-gcc (Arch Linux Repositories) 15.2.0
  CROSS prefix: riscv64-elf-
✓ PyTorch 2.10.0+cu128
✓ torchvision


## 2. 训练 Small MLP 并量化

如果 `small_mlp_data/` 已存在则跳过训练。

训练脚本 `small_mlp_quantize.py` 位于 `picorv32/fw/`，输出到同目录下的 `small_mlp_data/`。

In [3]:
if not os.path.isdir(DATA_DIR):
    print(f"训练模型并导出到 {DATA_DIR}/ ...")
    ret = subprocess.run(
        ["python3", "small_mlp_quantize.py",
         "--output-dir", DATA_DIR_REL,
         "--num-test", str(N_IMAGES),
         "--seed", "42"],
        cwd=FW_DIR,
        text=True
    )
    assert ret.returncode == 0, "训练失败!"
else:
    print(f"{DATA_DIR}/ 已存在, 跳过训练")

# 列出文件
print()
for f in sorted(os.listdir(DATA_DIR)):
    full = os.path.join(DATA_DIR, f)
    if os.path.isfile(full):
        print(f"  {f:30s} {os.path.getsize(full):>8d} B")
    else:
        n = len(os.listdir(full))
        print(f"  {f + '/':30s} ({n} files)")

训练模型并导出到 /home/jiao/git/INT8-CIM-of-jiao/picorv32/fw/small_mlp_data/ ...
Training SmallMLP (784→16→10)
  Epoch 1/20: loss=0.6961, acc=91.16%
  Epoch 2/20: loss=0.3143, acc=91.99%
  Epoch 3/20: loss=0.2738, acc=92.58%
  Epoch 4/20: loss=0.2474, acc=93.16%
  Epoch 5/20: loss=0.2299, acc=93.47%
  Epoch 6/20: loss=0.2166, acc=93.73%
  Epoch 7/20: loss=0.2055, acc=94.07%
  Epoch 8/20: loss=0.1963, acc=94.21%
  Epoch 9/20: loss=0.1880, acc=94.43%
  Epoch 10/20: loss=0.1819, acc=94.44%
  Epoch 11/20: loss=0.1753, acc=94.54%
  Epoch 12/20: loss=0.1708, acc=94.72%
  Epoch 13/20: loss=0.1651, acc=94.75%
  Epoch 14/20: loss=0.1605, acc=94.77%
  Epoch 15/20: loss=0.1571, acc=94.91%
  Epoch 16/20: loss=0.1531, acc=94.81%
  Epoch 17/20: loss=0.1503, acc=94.93%
  Epoch 18/20: loss=0.1473, acc=95.02%
  Epoch 19/20: loss=0.1440, acc=95.29%
  Epoch 20/20: loss=0.1423, acc=95.15%

Quantizing...
  FC1: s_w=0.010215, s_out=0.268343, mult=10, shift=16
  FC2: s_w=0.012717, s_out=0.332514, mult=673, shift=16


## 3. Golden Model 验证 (20 张图)

用 Python bit-accurate INT8 推理，和硬件行为完全一致。
结果保存到 `golden_results` 列表，后续写入文件。

In [4]:
def read_hex_u32(path):
    with open(path) as f:
        return [int(l.strip(), 16) for l in f if l.strip()]

def read_hex_u8(path):
    with open(path) as f:
        return [int(l.strip(), 16) & 0xFF for l in f if l.strip()]

# 量化参数
qp = read_hex_u32(os.path.join(DATA_DIR, "quant_params.hex"))
zps = read_hex_u32(os.path.join(DATA_DIR, "zero_points.hex"))
fc1_mult, fc1_shift = qp[0], qp[1]
fc2_mult, fc2_shift = qp[2], qp[3]
hw_zp1 = np.int32(np.uint32(zps[0]))
hw_zp2 = np.int32(np.uint32(zps[1]))

# 解包权重
def unpack_weights(chunk_path, out_dim, in_dim):
    chunks = read_hex_u32(chunk_path)
    w = np.zeros((out_dim, in_dim), dtype=np.int8)
    idx = 0
    for ob in range((out_dim + 15) // 16):
        for ib in range((in_dim + 15) // 16):
            for ch in range(64):
                r, cg = ch // 4, ch % 4
                word = chunks[idx]; idx += 1
                for b in range(4):
                    oi = ob * 16 + r
                    ii = ib * 16 + cg * 4 + b
                    if oi < out_dim and ii < in_dim:
                        val = (word >> (b*8)) & 0xFF
                        w[oi, ii] = np.int8(struct.unpack('b', bytes([val]))[0])
    return w

def unpack_bias(path):
    return np.array([np.int32(np.uint32(v)) for v in read_hex_u32(path)], dtype=np.int32)

def hw_mvm(x_u8, w_i8, b_i32, zp, mult, shift, relu):
    x_eff = np.clip(x_u8.astype(np.int32) - zp, 0, 511)
    acc = w_i8.astype(np.int32) @ x_eff + b_i32
    if relu: acc = np.maximum(acc, 0)
    out = np.zeros(len(acc), dtype=np.int8)
    for i in range(len(acc)):
        prod = int(acc[i]) * int(mult)
        shifted = (prod + (1 << (shift-1))) >> shift if shift > 0 else prod
        out[i] = np.int8(max(-128, min(127, shifted)))
    return out

w1 = unpack_weights(os.path.join(DATA_DIR, "fc1_weight_tiles.hex"), 16, 784)
b1 = unpack_bias(os.path.join(DATA_DIR, "fc1_bias.hex"))
w2 = unpack_weights(os.path.join(DATA_DIR, "fc2_weight_tiles.hex"), 10, 16)
b2 = unpack_bias(os.path.join(DATA_DIR, "fc2_bias.hex"))

print(f"FC1: w={w1.shape}, b={b1.shape}, mult={fc1_mult}, shift={fc1_shift}, zp={hw_zp1}")
print(f"FC2: w={w2.shape}, b={b2.shape}, mult={fc2_mult}, shift={fc2_shift}, zp={hw_zp2}")
print()

golden_results = []
for i in range(N_IMAGES):
    prefix = f"img_{i:04d}"
    img_path = os.path.join(DATA_DIR, "test_images", f"{prefix}.hex")
    lbl_path = os.path.join(DATA_DIR, "test_images", f"{prefix}_label.txt")
    img = np.array(read_hex_u8(img_path), dtype=np.uint8)
    label = int(open(lbl_path).read().strip())
    fc1 = hw_mvm(img, w1, b1, hw_zp1, fc1_mult, fc1_shift, True)
    fc2 = hw_mvm(fc1.view(np.uint8), w2, b2, hw_zp2, fc2_mult, fc2_shift, False)
    pred = int(np.argmax(fc2))
    golden_results.append((i, label, pred, pred == label))
    print(f"  [{i:04d}] label={label}, pred={pred} {'✓' if pred==label else '✗'}")

correct = sum(1 for r in golden_results if r[3])
print(f"\nGolden model 准确率: {correct}/{N_IMAGES} ({100*correct/N_IMAGES:.0f}%)")

FC1: w=(16, 784), b=(16,), mult=10, shift=16, zp=0
FC2: w=(10, 16), b=(10,), mult=673, shift=16, zp=0

  [0000] label=7, pred=7 ✓
  [0001] label=2, pred=2 ✓
  [0002] label=1, pred=1 ✓
  [0003] label=0, pred=0 ✓
  [0004] label=4, pred=4 ✓
  [0005] label=1, pred=1 ✓
  [0006] label=4, pred=4 ✓
  [0007] label=9, pred=9 ✓
  [0008] label=5, pred=6 ✗
  [0009] label=9, pred=9 ✓
  [0010] label=0, pred=0 ✓
  [0011] label=6, pred=6 ✓
  [0012] label=9, pred=9 ✓
  [0013] label=0, pred=0 ✓
  [0014] label=1, pred=1 ✓
  [0015] label=5, pred=5 ✓
  [0016] label=9, pred=9 ✓
  [0017] label=7, pred=7 ✓
  [0018] label=3, pred=8 ✗
  [0019] label=4, pred=4 ✓
  [0020] label=9, pred=9 ✓
  [0021] label=6, pred=6 ✓
  [0022] label=6, pred=6 ✓
  [0023] label=5, pred=5 ✓
  [0024] label=4, pred=4 ✓
  [0025] label=0, pred=0 ✓
  [0026] label=7, pred=7 ✓
  [0027] label=4, pred=4 ✓
  [0028] label=0, pred=0 ✓
  [0029] label=1, pred=1 ✓
  [0030] label=3, pred=3 ✓
  [0031] label=1, pred=1 ✓
  [0032] label=3, pred=3 ✓
  [003

## 4. 批量编译 20 个 firmware.hex

对每张测试图，调用 `make` 编译出独立的 `firmware.hex`，
复制到 `sw/fw_hex_batch/firmware_XXXX.hex`。

> `make` 在 `picorv32/fw/` 下运行：
> `gen_fw_data.py` → `model_data.c` → GCC 交叉编译 → `firmware.hex`

In [5]:
assert cross_prefix is not None, "RISC-V GCC 未找到，无法编译!"

os.makedirs(OUT_DIR, exist_ok=True)
fw_hex = os.path.join(FW_DIR, "firmware.hex")

compile_ok = 0
compile_fail = 0

for i in range(N_IMAGES):
    idx = f"{i:04d}"
    label = golden_results[i][1]

    # clean
    subprocess.run(["make", "-s", "clean"], check=False,
                   capture_output=True, cwd=FW_DIR)

    # compile (DATA_DIR_REL 是相对于 FW_DIR 的路径)
    ret = subprocess.run(
        ["make", "-s",
         f"DATA_DIR={DATA_DIR_REL}",
         f"IMAGE_IDX={i}",
         f"CROSS={cross_prefix}"],
        capture_output=True, text=True, cwd=FW_DIR
    )

    if ret.returncode != 0 or not os.path.exists(fw_hex):
        print(f"  [{idx}] ✗ 编译失败!")
        if ret.stderr.strip():
            print(f"         {ret.stderr.strip()[:200]}")
        compile_fail += 1
        continue

    out_path = os.path.join(OUT_DIR, f"firmware_{idx}.hex")
    shutil.copy(fw_hex, out_path)
    n_words = sum(1 for l in open(fw_hex) if l.strip())
    print(f"  [{idx}] ✓ label={label}, {n_words} words ({n_words*4} bytes)")
    compile_ok += 1

print(f"\n编译完成: {compile_ok} 成功, {compile_fail} 失败")

  [0000] ✓ label=7, 3723 words (14892 bytes)
  [0001] ✓ label=2, 3723 words (14892 bytes)
  [0002] ✓ label=1, 3723 words (14892 bytes)
  [0003] ✓ label=0, 3723 words (14892 bytes)
  [0004] ✓ label=4, 3723 words (14892 bytes)
  [0005] ✓ label=1, 3723 words (14892 bytes)
  [0006] ✓ label=4, 3723 words (14892 bytes)
  [0007] ✓ label=9, 3723 words (14892 bytes)
  [0008] ✓ label=5, 3723 words (14892 bytes)
  [0009] ✓ label=9, 3723 words (14892 bytes)
  [0010] ✓ label=0, 3723 words (14892 bytes)
  [0011] ✓ label=6, 3723 words (14892 bytes)
  [0012] ✓ label=9, 3723 words (14892 bytes)
  [0013] ✓ label=0, 3723 words (14892 bytes)
  [0014] ✓ label=1, 3723 words (14892 bytes)
  [0015] ✓ label=5, 3723 words (14892 bytes)
  [0016] ✓ label=9, 3723 words (14892 bytes)
  [0017] ✓ label=7, 3723 words (14892 bytes)
  [0018] ✓ label=3, 3723 words (14892 bytes)
  [0019] ✓ label=4, 3723 words (14892 bytes)
  [0020] ✓ label=9, 3723 words (14892 bytes)
  [0021] ✓ label=6, 3723 words (14892 bytes)
  [0022] ✓

## 4.5 保存 Golden Results + Labels

将 golden model 推理结果写入两个位置：
- `sw/fw_hex_batch/golden_results.txt` — 供 VCS 仿真对比
- `picorv32/fw/small_mlp_data/golden_results.txt` — 供 PYNQ 端对比

同时生成 `labels.txt` 供 VCS 批量仿真脚本使用。

In [6]:
def write_golden(path):
    with open(path, "w") as f:
        f.write("# Golden Model Results (bit-accurate INT8, small_mlp 784->16->10)\n")
        f.write(f"# Generated by prepare_picorv32_env.ipynb\n")
        f.write(f"# Quant: fc1=({fc1_mult},{fc1_shift}), fc2=({fc2_mult},{fc2_shift})\n")
        f.write(f"# ZP: fc1={hw_zp1}, fc2={hw_zp2}\n")
        f.write(f"# Format: idx label pred correct\n")
        for i, label, pred, ok in golden_results:
            f.write(f"{i} {label} {pred} {1 if ok else 0}\n")
    print(f"  Saved: {path}")

# 写入 fw_hex_batch/
write_golden(os.path.join(OUT_DIR, "golden_results.txt"))

# 同时写入 small_mlp_data/ (方便上传到 PYNQ 时一起带上)
write_golden(os.path.join(DATA_DIR, "golden_results.txt"))

# labels.txt (供 VCS 批量仿真脚本解析)
labels_path = os.path.join(OUT_DIR, "labels.txt")
with open(labels_path, "w") as f:
    f.write("# idx label\n")
    for i, label, pred, ok in golden_results:
        f.write(f"{i:04d} {label}\n")
print(f"  Saved: {labels_path}")

correct = sum(1 for r in golden_results if r[3])
print(f"\n  {N_IMAGES} images, accuracy {correct}/{N_IMAGES} ({100*correct/N_IMAGES:.0f}%)")
print()
print("golden_results.txt 内容预览:")
with open(os.path.join(OUT_DIR, "golden_results.txt")) as f:
    print(f.read())

  Saved: /home/jiao/git/INT8-CIM-of-jiao/sw/fw_hex_batch/golden_results.txt
  Saved: /home/jiao/git/INT8-CIM-of-jiao/picorv32/fw/small_mlp_data/golden_results.txt
  Saved: /home/jiao/git/INT8-CIM-of-jiao/sw/fw_hex_batch/labels.txt

  200 images, accuracy 193/200 (96%)

golden_results.txt 内容预览:
# Golden Model Results (bit-accurate INT8, small_mlp 784->16->10)
# Generated by prepare_picorv32_env.ipynb
# Quant: fc1=(10,16), fc2=(673,16)
# ZP: fc1=0, fc2=0
# Format: idx label pred correct
0 7 7 1
1 2 2 1
2 1 1 1
3 0 0 1
4 4 4 1
5 1 1 1
6 4 4 1
7 9 9 1
8 5 6 0
9 9 9 1
10 0 0 1
11 6 6 1
12 9 9 1
13 0 0 1
14 1 1 1
15 5 5 1
16 9 9 1
17 7 7 1
18 3 8 0
19 4 4 1
20 9 9 1
21 6 6 1
22 6 6 1
23 5 5 1
24 4 4 1
25 0 0 1
26 7 7 1
27 4 4 1
28 0 0 1
29 1 1 1
30 3 3 1
31 1 1 1
32 3 3 1
33 4 4 1
34 7 7 1
35 2 2 1
36 7 7 1
37 1 1 1
38 2 2 1
39 1 1 1
40 1 1 1
41 7 7 1
42 4 4 1
43 2 2 1
44 3 3 1
45 5 5 1
46 1 1 1
47 2 2 1
48 4 4 1
49 4 4 1
50 6 6 1
51 3 3 1
52 5 5 1
53 5 5 1
54 6 6 1
55 0 0 1
56 4 4 1
57 1 1 

## 5. 检查 BRAM 占用

In [7]:
fw0 = os.path.join(OUT_DIR, "firmware_0000.hex")
if os.path.exists(fw0):
    n_words = sum(1 for l in open(fw0) if l.strip())
    n_bytes = n_words * 4
    print(f"firmware.hex: {n_words} words = {n_bytes} bytes ({n_bytes/1024:.1f} KB)")
    print(f"BRAM 容量:    8192 words = 32768 bytes (32.0 KB)")
    print(f"占用率:       {100*n_bytes/32768:.1f}%")
    print()
    if n_bytes > 32768:
        print("⚠ 超出 32KB BRAM!")
    else:
        print("✓ 固件可以放入 BRAM")
else:
    print("⚠ firmware_0000.hex 不存在，编译可能失败了")

firmware.hex: 3723 words = 14892 bytes (14.5 KB)
BRAM 容量:    8192 words = 32768 bytes (32.0 KB)
占用率:       45.4%

✓ 固件可以放入 BRAM


## 6. 输出总览

In [8]:
print("="*60)
print("生成的文件")
print("="*60)
print(f"\n{OUT_DIR}/")
if os.path.isdir(OUT_DIR):
    for f in sorted(os.listdir(OUT_DIR)):
        sz = os.path.getsize(os.path.join(OUT_DIR, f))
        print(f"  {f:35s} {sz:>8d} B")

print(f"\n{DATA_DIR}/")
gr = os.path.join(DATA_DIR, "golden_results.txt")
if os.path.exists(gr):
    print(f"  golden_results.txt               {os.path.getsize(gr):>8d} B  (新生成)")

print()
print("="*60)
print("下一步")
print("="*60)
print()
print("→ VCS 虚拟机 (仿真验证)")
print("  复制到虚拟机:")
print("    sw/fw_hex_batch/                  ← 20 个 firmware hex + golden_results.txt")
print("    picorv32/hw/                      ← RTL + tb + scripts")
print("    hw/rtl/                           ← CIM IP")
print("  运行:")
print("    cd picorv32/")
print("    bash hw/scripts/run_tb_rv32_batch.sh")
print()
print("→ PYNQ 板 (上板验证)")
print("  上传到 PYNQ Jupyter 同一目录:")
print("    cim_rv32_soc.bit                  ← bitstream")
print("    cim_rv32_soc.hwh                  ← 硬件描述 (必须和 .bit 同名)")
print("    small_mlp_data/                   ← 模型数据 + golden_results.txt")
print("    pynq_verify_rv32.ipynb            ← 验证 notebook")
print()
print("  注意: bitstream 烧进 BRAM 的 firmware.hex 是 IMAGE_IDX=0 那个。")
print("  加载 bitstream 后 PicoRV32 立即执行，结果写入 Result BRAM。")
print("  PS 通过 MMIO 读 0x4000_0000 即可拿到推理结果。")

生成的文件

/home/jiao/git/INT8-CIM-of-jiao/sw/fw_hex_batch/
  firmware_0000.hex                      33507 B
  firmware_0001.hex                      33507 B
  firmware_0002.hex                      33507 B
  firmware_0003.hex                      33507 B
  firmware_0004.hex                      33507 B
  firmware_0005.hex                      33507 B
  firmware_0006.hex                      33507 B
  firmware_0007.hex                      33507 B
  firmware_0008.hex                      33507 B
  firmware_0009.hex                      33507 B
  firmware_0010.hex                      33507 B
  firmware_0011.hex                      33507 B
  firmware_0012.hex                      33507 B
  firmware_0013.hex                      33507 B
  firmware_0014.hex                      33507 B
  firmware_0015.hex                      33507 B
  firmware_0016.hex                      33507 B
  firmware_0017.hex                      33507 B
  firmware_0018.hex                      33507 B
  firmware_00